# PriorityLab — M1 TF-IDF + Logistic Regression

**Goal:** train Dummy + TF-IDF/LogReg baselines, evaluate on held-out data, inspect top terms.

**Prereq:** finish M0 EDA go/no-go first.

**Rule:** do not treat printed metrics as real until **you** run this notebook. Cells below call live training.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data import load_prepared_splits
from src.evaluate import evaluate_model, plot_confusion_matrix, save_metrics
from src.explain import top_global_terms, top_terms_for_text, format_term_list
from src.train import fit_baselines, save_model, MODEL_VERSION

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
MODELS = ROOT / "models"
FIGURES.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)

## 1. Load prepared stratified splits

Uses subject+body text only. Exact duplicate texts are dropped (keep first).

In [ ]:
train_df, val_df, test_df = load_prepared_splits()
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, part.shape, "important rate", round(part["important"].mean(), 4))

X_train, y_train = train_df["text"], train_df["important"]
X_val, y_val = val_df["text"], val_df["important"]
X_test, y_test = test_df["text"], test_df["important"]

## 2. Fit baselines (train fold only)

In [ ]:
models = fit_baselines(X_train, y_train)
print("fitted:", list(models))

## 3. Evaluate on validation (tune/compare here)

Keep the test fold untouched until a final comparison.

In [ ]:
val_scores = {}
for name, model in models.items():
    metrics = evaluate_model(model, X_val, y_val)
    val_scores[name] = metrics
    print("\n===", name, "(validation) ===")
    print("accuracy", round(metrics["accuracy"], 4))
    print("macro_f1", round(metrics["macro_f1"], 4))
    print("recall_important", round(metrics["recall_important"], 4))
    print(metrics["classification_report"])

## 4. Final test metrics + save artifacts

In [ ]:
primary = models["tfidf_logreg"]
test_metrics = evaluate_model(primary, X_test, y_test)
print(test_metrics["classification_report"])

cm_path = plot_confusion_matrix(
    test_metrics["confusion_matrix"],
    FIGURES / "m1_tfidf_logreg_confusion.png",
    title="TF-IDF + LogReg (test)",
)
metrics_path = save_metrics(
    {
        "model_version": MODEL_VERSION,
        "validation": {k: {kk: vv for kk, vv in v.items() if kk != "classification_report"} for k, v in val_scores.items()},
        "test": {k: v for k, v in test_metrics.items() if k != "classification_report"},
        "test_classification_report": test_metrics["classification_report"],
    },
    REPORTS / "m1_metrics.json",
)
model_path = save_model(
    primary,
    MODELS,
    metadata={
        "features": "subject+body text",
        "vectorizer": "TfidfVectorizer ngram=(1,2)",
        "classifier": "LogisticRegression class_weight=balanced",
    },
)
print("saved", cm_path)
print("saved", metrics_path)
print("saved", model_path)

## 5. Explainability (model evidence, not human reasons)

In [ ]:
print("Top global terms (positive class coefficients):")
for term, weight in top_global_terms(primary, k=15):
    print(f"  {term:30s} {weight:+.4f}")

example = "subject: Please submit the science report by 08:00 body: Upload your PDF before the deadline."
local = top_terms_for_text(primary, example, k=8)
print("\nExample message top terms:", format_term_list(local))
for term, score in local:
    print(f"  {term:30s} {score:+.4f}")

## 6. Error peek on validation

Inspect a few false negatives (missed important) and false positives.

In [ ]:
val_pred = primary.predict(X_val)
val_view = val_df.copy()
val_view["pred"] = val_pred

fn = val_view[(val_view["important"] == 1) & (val_view["pred"] == 0)].head(5)
fp = val_view[(val_view["important"] == 0) & (val_view["pred"] == 1)].head(5)
print("False negatives (important missed):", len(fn))
display(fn[["subject_clean", "priority_norm", "important", "pred"]])
print("False positives (flagged but not important):", len(fp))
display(fp[["subject_clean", "priority_norm", "important", "pred"]])